# Step 8 — Model Comparison

## Load Evaluation Test Dataset

In [3]:
import json
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.job_item import JobItem

with open("../data/fine_tune/test_set.json") as f:
    data = json.load(f)

items = [JobItem(**row) for row in data]

for item in items:
    item.make_prompt()

print("Evaluation samples:", len(items))

Evaluation samples: 90


## Generate Predictions

In [25]:
from openai import OpenAI
import numpy as np
import re

client = OpenAI()

base_model = "gpt-4.1-nano"
fine_model = "ft:gpt-3.5-turbo-0125:personal:salarysense:DG1gFFGr"

true = []
base_pred = []
ft_pred = []

for item in items:

    true.append(item.salary)

    base = client.chat.completions.create(
        model=base_model,
        messages=[{"role":"user","content":item.prompt}],
        temperature=0
    )

    ft = client.chat.completions.create(
        model=fine_model,
        messages=[{"role":"user","content":item.prompt}],
        temperature=0
    )

    def extract_number(text):
        text = text.replace(",", "")
        
        match = re.search(r"\d{4,6}", text)
        
        if match:
            return float(match.group())
        
        return np.nan
    
    base_pred.append(extract_number(base.choices[0].message.content))
    ft_pred.append(extract_number(ft.choices[0].message.content))

## Compute Error 

In [26]:
mae_base = np.nanmean(np.abs(np.array(true) - np.array(base_pred)))
mae_ft = np.nanmean(np.abs(np.array(true) - np.array(ft_pred)))

print("Base Model MAE:", mae_base)
print("Fine-tuned Model MAE:", mae_ft)

Base Model MAE: 820.0333333333333
Fine-tuned Model MAE: 986.064606741573


## Attractive Plotly Chart (Visualization)

In [27]:
import plotly.graph_objects as go

results = [
    ("Base Model", "orange", mae_base),
    ("Fine-tuned Model", "red", mae_ft)
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(
    x=labels,
    y=values,
    marker_color=colors
))

fig.update_layout(
    title="Fine-Tuned Model Performance Improvement",
    yaxis=dict(title="Mean Absolute Error"),
    xaxis=dict(tickangle=-20),
    width=900,
    height=600
)

fig.show()

## standard ML visualization (Better Visualization)

In [29]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=true,
    y=base_pred,
    mode='markers',
    name='Base Model'
))

fig.add_trace(go.Scatter(
    x=true,
    y=ft_pred,
    mode='markers',
    name='Fine-Tuned Model'
))

fig.add_trace(go.Scatter(
    x=[0,500000],
    y=[0,500000],
    mode='lines',
    name='Perfect Prediction'
))

fig.update_layout(
    title="Actual vs Predicted Salary",
    xaxis_title="True Salary",
    yaxis_title="Predicted Salary",
    width=900,
    height=700
)

fig.show()

## Run the predictor on Gradio Interface

In [10]:
import gradio as gr
from openai import OpenAI

client = OpenAI()

MODEL = "ft:gpt-3.5-turbo-0125:personal:salarysense:DG1gFFGr"


def predict_salary(job_description):

    prompt = f"""
    Estimate the salary for the following job.

    {job_description}

    Salary is $
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return "$" + response.choices[0].message.content.strip()

demo = gr.Interface(
    fn=predict_salary,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Enter job description..."
    ),
    outputs="text",
    title="AI Salary Predictor",
    description="Predict salary from job description using a fine-tuned LLM."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
